# Model Training & Evaluation
## CW1 - XGBoost vs LightGBM Deep Comparison

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import cross_val_score, cross_val_predict, KFold, learning_curve
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import xgboost as xgb
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

np.random.seed(123)

trn = pd.read_csv('../data/raw/CW1_train.csv')
tst = pd.read_csv('../data/raw/CW1_test.csv')

## 1. Preprocessing

In [ ]:
def preprocess(df):
    df = df.copy()
    
    for col in ['x', 'y', 'z']:
        df[col] = df[col].replace(0, np.nan)
        df[col] = df[col].fillna(df[col].median())
    
    cut_map = {'Fair': 0, 'Good': 1, 'Very Good': 2, 'Premium': 3, 'Ideal': 4}
    color_map = {'J': 0, 'I': 1, 'H': 2, 'G': 3, 'F': 4, 'E': 5, 'D': 6}
    clarity_map = {'I1': 0, 'SI2': 1, 'SI1': 2, 'VS2': 3, 'VS1': 4, 'VVS2': 5, 'VVS1': 6, 'IF': 7}
    
    df['cut_ord'] = df['cut'].map(cut_map)
    df['color_ord'] = df['color'].map(color_map)
    df['clarity_ord'] = df['clarity'].map(clarity_map)
    
    df['depth_sq'] = df['depth'] ** 2
    df['depth_cb'] = df['depth'] ** 3
    df['depth_x_b3'] = df['depth'] * df['b3']
    df['depth_x_b1'] = df['depth'] * df['b1']
    df['depth_x_a1'] = df['depth'] * df['a1']
    df['depth_x_a4'] = df['depth'] * df['a4']
    df['depth_x_table'] = df['depth'] * df['table']
    df['b3_x_b1'] = df['b3'] * df['b1']
    df['b3_x_a1'] = df['b3'] * df['a1']
    df['a1_x_a4'] = df['a1'] * df['a4']
    df['b1_x_a1'] = df['b1'] * df['a1']
    df['b3_sq'] = df['b3'] ** 2
    df['b1_sq'] = df['b1'] ** 2
    df['a1_sq'] = df['a1'] ** 2
    df['a4_sq'] = df['a4'] ** 2
    df['table_sq'] = df['table'] ** 2
    df['volume'] = df['x'] * df['y'] * df['z']
    df['log_carat'] = np.log1p(df['carat'])
    df['log_price'] = np.log1p(df['price'])
    df['xy_ratio'] = df['x'] / df['y'].replace(0, np.nan).fillna(df['y'].median())
    
    df = df.drop(columns=['cut', 'color', 'clarity'])
    return df

trn_proc = preprocess(trn)
tst_proc = preprocess(tst)

X_trn = trn_proc.drop(columns=['outcome'])
y_trn = trn_proc['outcome']
X_tst = tst_proc

common_cols = X_trn.columns.intersection(X_tst.columns)
X_trn = X_trn[common_cols]
X_tst = X_tst[common_cols]

print(f"Features: {X_trn.shape[1]}")

## 2. Hyperparameter Tuning

Testing multiple configurations for both XGBoost and LightGBM.

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=123)

xgb_configs = {
    'XGB Default': {'n_estimators': 200, 'max_depth': 5, 'learning_rate': 0.1},
    'XGB Shallow+Slow': {'n_estimators': 1500, 'max_depth': 3, 'learning_rate': 0.02, 'subsample': 0.8, 'colsample_bytree': 0.75, 'reg_alpha': 1.0, 'reg_lambda': 3.0},
    'XGB Deep+Fast': {'n_estimators': 500, 'max_depth': 7, 'learning_rate': 0.05, 'subsample': 0.85, 'colsample_bytree': 0.8},
    'XGB Regularised': {'n_estimators': 700, 'max_depth': 3, 'learning_rate': 0.015, 'subsample': 0.9, 'colsample_bytree': 0.9, 'min_child_weight': 2, 'gamma': 0.1, 'reg_alpha': 0.05, 'reg_lambda': 2.0},
    'XGB Medium': {'n_estimators': 1000, 'max_depth': 4, 'learning_rate': 0.03, 'subsample': 0.8, 'colsample_bytree': 0.7, 'reg_alpha': 0.5, 'reg_lambda': 2.0},
}

print(f"{'Config':25s}  {'Mean R²':>10s}  {'Std':>8s}")
print("-" * 48)
xgb_results = {}
for name, cfg in xgb_configs.items():
    m = xgb.XGBRegressor(**cfg, random_state=123, n_jobs=-1)
    scores = cross_val_score(m, X_trn, y_trn, cv=kf, scoring='r2', n_jobs=1)
    xgb_results[name] = scores
    print(f"{name:25s}  {scores.mean():10.4f}  {scores.std():8.4f}")

In [ ]:
lgb_configs = {
    'LGB Default': {'n_estimators': 200, 'max_depth': 5, 'learning_rate': 0.1},
    'LGB Shallow+Slow': {'n_estimators': 1500, 'max_depth': 3, 'learning_rate': 0.02, 'subsample': 0.8, 'colsample_bytree': 0.75, 'reg_alpha': 1.0, 'reg_lambda': 3.0, 'num_leaves': 31},
    'LGB Deep': {'n_estimators': 1000, 'max_depth': 8, 'learning_rate': 0.03, 'subsample': 0.9, 'colsample_bytree': 0.85, 'num_leaves': 100},
    'LGB Regularised': {'n_estimators': 700, 'max_depth': 3, 'num_leaves': 47, 'learning_rate': 0.01, 'subsample': 0.9, 'colsample_bytree': 0.7, 'min_child_samples': 10, 'reg_alpha': 0.2, 'reg_lambda': 5.0},
    'LGB Medium': {'n_estimators': 1000, 'max_depth': 4, 'learning_rate': 0.03, 'subsample': 0.8, 'colsample_bytree': 0.7, 'reg_alpha': 0.5, 'reg_lambda': 2.0, 'num_leaves': 31},
}

print(f"{'Config':25s}  {'Mean R²':>10s}  {'Std':>8s}")
print("-" * 48)
lgb_results = {}
for name, cfg in lgb_configs.items():
    m = lgb.LGBMRegressor(**cfg, random_state=123, n_jobs=-1, verbose=-1)
    scores = cross_val_score(m, X_trn, y_trn, cv=kf, scoring='r2', n_jobs=1)
    lgb_results[name] = scores
    print(f"{name:25s}  {scores.mean():10.4f}  {scores.std():8.4f}")

## 3. Tuning Results Visualisation

In [ ]:
all_results = {**xgb_results, **lgb_results}

fig, ax = plt.subplots(figsize=(10, 6))
names = list(all_results.keys())
means = [all_results[n].mean() for n in names]
stds = [all_results[n].std() for n in names]

colors = ['#e74c3c' if 'XGB' in n else '#3498db' for n in names]
best_idx = means.index(max(means))
colors[best_idx] = '#2ecc71'

bars = ax.barh(names, means, xerr=stds, color=colors, edgecolor='black', alpha=0.8, capsize=4)
ax.set_xlabel('R² Score (5-Fold CV)')
ax.set_title('XGBoost vs LightGBM - Hyperparameter Comparison')

for i, (m, s) in enumerate(zip(means, stds)):
    ax.text(m + s + 0.003, i, f'{m:.4f}', va='center', fontsize=8)

ax.legend(handles=[
    plt.Rectangle((0,0),1,1, fc='#e74c3c', alpha=0.8, label='XGBoost'),
    plt.Rectangle((0,0),1,1, fc='#3498db', alpha=0.8, label='LightGBM'),
    plt.Rectangle((0,0),1,1, fc='#2ecc71', alpha=0.8, label='Best'),
], labels=['XGBoost', 'LightGBM', 'Best'])

plt.tight_layout()
plt.savefig('../reports/figures/xgb_vs_lgb_tuning.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Best Models - Detailed Comparison

In [ ]:
xgb_best = xgb.XGBRegressor(
    n_estimators=700, max_depth=3, learning_rate=0.015,
    subsample=0.9, colsample_bytree=0.9, min_child_weight=2,
    gamma=0.1, reg_alpha=0.05, reg_lambda=2.0,
    random_state=123, n_jobs=-1
)

lgb_best = lgb.LGBMRegressor(
    n_estimators=700, max_depth=3, num_leaves=47,
    learning_rate=0.01, subsample=0.9, colsample_bytree=0.7,
    min_child_samples=10, reg_alpha=0.2, reg_lambda=5.0,
    random_state=123, n_jobs=-1, verbose=-1
)

xgb_oof = cross_val_predict(xgb_best, X_trn, y_trn, cv=kf, n_jobs=1)
lgb_oof = cross_val_predict(lgb_best, X_trn, y_trn, cv=kf, n_jobs=1)

print("=== Out-of-Fold Metrics ===")
print(f"{'Metric':15s}  {'XGBoost':>10s}  {'LightGBM':>10s}")
print("-" * 40)
print(f"{'R²':15s}  {r2_score(y_trn, xgb_oof):10.4f}  {r2_score(y_trn, lgb_oof):10.4f}")
print(f"{'RMSE':15s}  {np.sqrt(mean_squared_error(y_trn, xgb_oof)):10.4f}  {np.sqrt(mean_squared_error(y_trn, lgb_oof)):10.4f}")
print(f"{'MAE':15s}  {mean_absolute_error(y_trn, xgb_oof):10.4f}  {mean_absolute_error(y_trn, lgb_oof):10.4f}")

## 5. Residual Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for i, (name, oof) in enumerate([('XGBoost', xgb_oof), ('LightGBM', lgb_oof)]):
    residuals = y_trn - oof
    
    axes[0, i].scatter(oof, residuals, alpha=0.2, s=5)
    axes[0, i].axhline(y=0, color='red', linestyle='--')
    axes[0, i].set_title(f'{name} - Residuals vs Predicted')
    axes[0, i].set_xlabel('Predicted')
    axes[0, i].set_ylabel('Residual')
    
    axes[1, i].hist(residuals, bins=50, edgecolor='black', alpha=0.7)
    axes[1, i].set_title(f'{name} - Residual Distribution')
    axes[1, i].set_xlabel('Residual')

plt.tight_layout()
plt.savefig('../reports/figures/residual_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Actual vs Predicted

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for i, (name, oof) in enumerate([('XGBoost', xgb_oof), ('LightGBM', lgb_oof)]):
    axes[i].scatter(y_trn, oof, alpha=0.2, s=5)
    mn, mx = y_trn.min(), y_trn.max()
    axes[i].plot([mn, mx], [mn, mx], 'r--', linewidth=2)
    axes[i].set_title(f'{name} - Actual vs Predicted (R²={r2_score(y_trn, oof):.4f})')
    axes[i].set_xlabel('Actual')
    axes[i].set_ylabel('Predicted')

plt.tight_layout()
plt.savefig('../reports/figures/actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Feature Importance Comparison

In [ ]:
xgb_best.fit(X_trn, y_trn)
lgb_best.fit(X_trn, y_trn)

xgb_imp = pd.Series(xgb_best.feature_importances_, index=X_trn.columns).sort_values(ascending=False).head(15)
lgb_imp = pd.Series(lgb_best.feature_importances_, index=X_trn.columns).sort_values(ascending=False).head(15)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

xgb_imp.plot(kind='barh', ax=axes[0], color='#e74c3c', edgecolor='black', alpha=0.8)
axes[0].set_title('XGBoost - Top 15 Features')
axes[0].set_xlabel('Importance')
axes[0].invert_yaxis()

lgb_imp.plot(kind='barh', ax=axes[1], color='#3498db', edgecolor='black', alpha=0.8)
axes[1].set_title('LightGBM - Top 15 Features')
axes[1].set_xlabel('Importance')
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig('../reports/figures/feature_importance_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Blended Model

In [ ]:
blend_weights = [(0.3, 0.7), (0.4, 0.6), (0.5, 0.5), (0.6, 0.4), (0.7, 0.3)]

print(f"{'XGB Weight':>12s}  {'LGB Weight':>12s}  {'Blended R²':>12s}")
print("-" * 40)
for w_xgb, w_lgb in blend_weights:
    blend_oof = w_xgb * xgb_oof + w_lgb * lgb_oof
    r2 = r2_score(y_trn, blend_oof)
    print(f"{w_xgb:12.1f}  {w_lgb:12.1f}  {r2:12.4f}")

## 9. Final Predictions

In [ ]:
blend_w_xgb = 0.40
blend_w_lgb = 0.60

yhat_xgb = xgb_best.predict(X_tst)
yhat_lgb = lgb_best.predict(X_tst)
yhat = blend_w_xgb * yhat_xgb + blend_w_lgb * yhat_lgb

out = pd.DataFrame({'yhat': yhat})
out.to_csv('../CW1_submission_K23115695.csv', index=False)
print(f"Predictions saved!")
print(f"Mean: {yhat.mean():.3f}, Std: {yhat.std():.3f}")

## Summary

- Both XGBoost and LightGBM achieve similar performance (R² ≈ 0.477)
- Regularised configurations with shallow trees and slow learning rates perform best
- Blending at 40/60 (XGBoost/LightGBM) gives a small improvement in stability
- `depth`, `depth_sq`, and `depth_cb` are by far the most important features
- Final blended out-of-fold R² = 0.4775